# AI Engineer Challenge
Dot Indonesia

**Nama:** Gymnastiar Al K

---
Oke jadi gw milih bagian 1A (Object Tracking Mobil pake YOLO).
Notebook ini isinya implementasi + jawaban teori.

---
## Bagian 1A: Object Tracking Mobil pake YOLO

Gunain pre-trained YOLO buat deteksi + tracking mobil di video.

Stepnya:
- Install dependensi
- Load model YOLO
- Download video sample
- Deteksi + tracking tiap frame
- Visualisasi bounding box

In [ ]:
# install dulu semua yang dibutuhin
# YOLO pake ultralytics, plus opencv buat ngolah video
!pip install -q ultralytics opencv-python-headless matplotlib numpy tqdm

# kalo mau pake model yg lebih gede tinggal ganti, tapi yg nano dulu aja biar cepet
print("dependencies installed")

In [ ]:
# import2 yang diperlukan
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from IPython.display import display, Video
from tqdm.notebook import tqdm
import os
from pathlib import Path

print(f"torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")  # kalo false berarti pake CPU, agak lambat

### Load Model
YOLOv8n, yang paling ringan. Pre-trained di COCO, jadi udah tau kelas mobil, bus, truk, motor.

In [ ]:
# pake YOLOv8n aja biar cepet, kalo pengen lebih akurat bisa ganti ke yolov8m atau yolov8l
model = YOLO('yolov8n.pt')

# kelas kendaraan di COCO: 2=mobil, 3=motor, 5=bus, 7=truk
vehicle_classes = [2, 3, 5, 7]

print("model loaded:", 'yolov8n.pt')
print("vehicle classes:", [model.names[c] for c in vehicle_classes])

### Download Video Sample
Pake video dari repo ultralytics, isinya lalu lintas jalan.

In [ ]:
# download video sample
VIDEO_PATH = 'traffic.mp4'
!wget -q --show-progress -O $VIDEO_PATH https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.mp4

# cek filenya ada atau ngga
if os.path.exists(VIDEO_PATH) and os.path.getsize(VIDEO_PATH) > 10000:
    print(f"video downloaded: {os.path.getsize(VIDEO_PATH)/1024:.1f} KB")
else:
    print("gagal download, ganti link..")
    # fallback kalo link rusak
    !wget -q -O $VIDEO_PATH https://www.dropbox.com/scl/fi/bi7x1ouqv78ogbnwhvx0u/traffic.mp4?rlkey=h8y4d3r0tjx1q5kxn4z6m9f7p 2>/dev/null || echo "skip fallback"

### Fungsi Deteksi + Tracking
Bikin fungsi buat deteksi mobil tiap frame, plus tracking biar ID mobilnya konsisten.

In [ ]:
# fungsi utama: deteksi + tracking mobil di satu frame
def detect_vehicles(frame, model, conf=0.4):
    """
    deteksi kendaraan pake YOLO, return frame yang udah di-annotate sama list deteksi
    """
    # track dengan ByteTrack (biar ID nya konsisten antar frame)
    results = model.track(frame, persist=True, conf=conf, iou=0.5,
                         tracker='bytetrack.yaml', verbose=False)

    detections = []
    frame_out = frame.copy()

    if results[0].boxes is None:
        return frame_out, detections

    boxes = results[0].boxes.xyxy.cpu().numpy()
    confs = results[0].boxes.conf.cpu().numpy()
    cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)

    # cek kalo ada track IDs
    if results[0].boxes.id is not None:
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)
    else:
        track_ids = [-1] * len(boxes)

    for box, conf, cls_id, tid in zip(boxes, confs, cls_ids, track_ids):
        # skip kalo bukan kendaraan
        if cls_id not in vehicle_classes:
            continue

        x1, y1, x2, y2 = map(int, box)
        label = model.names[cls_id]

        detections.append({
            'bbox': (x1, y1, x2, y2),
            'conf': float(conf),
            'class': label,
            'track_id': int(tid)
        })

        # warna random based on track ID biar beda2 tiap mobil
        color = (tid * 127 % 255, tid * 63 % 255, tid * 255 % 255) if tid > 0 else (0, 255, 0)

        cv2.rectangle(frame_out, (x1, y1), (x2, y2), color, 2)

        # teks label: ID kalo ada
        tid_str = f"ID:{tid} " if tid > 0 else ""
        txt = f"{tid_str}{label} {conf:.2f}"

        # background biar teksnya kebaca
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
        cv2.rectangle(frame_out, (x1, y1-20), (x1+tw+5, y1), color, -1)
        cv2.putText(frame_out, txt, (x1+2, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    return frame_out, detections

print("fungsi siap")

### Proses Video
Loop tiap frame, deteksi + tracking, terus simpen jadi video baru.

In [ ]:
# proses video per frame
def process_video(video_path, model, output_name='output_tracked.mp4', conf=0.4, max_frames=300):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("gagal buka video")
        return None, []

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total = min(total, max_frames)

    out = cv2.VideoWriter(output_name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    counts = []

    print(f"proses {total} frames...")
    for i in tqdm(range(total)):
        ret, frame = cap.read()
        if not ret:
            break
        annotated, dets = detect_vehicles(frame, model, conf)
        cv2.putText(annotated, f"frame {i+1}/{total} | {len(dets)} kendaraan",
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
        out.write(annotated)
        counts.append(len(dets))

    cap.release()
    out.release()

    print(f"\nselesai! output: {output_name}")
    if counts:
        print(f"rata2 kendaraan per frame: {np.mean(counts):.1f}")
        print(f"paling banyak: {np.max(counts)} kendaraan dalam 1 frame")

    return output_name, counts

# jalanin
out_video, counts = process_video(VIDEO_PATH, model, 'output_tracked.mp4', 0.4, 300)

In [ ]:
# compress biar bisa ditampilin di colab
!ffmpeg -y -i output_tracked.mp4 -vcodec libx264 -crf 23 output_compressed.mp4 -loglevel error

# tampilin videonya
if os.path.exists('output_compressed.mp4'):
    display(Video('output_compressed.mp4', width=720))
elif os.path.exists('output_tracked.mp4'):
    display(Video('output_tracked.mp4', width=720))
else:
    print("video output not found :/")

### Sample Frame
Biar keliatan detail deteksinya di beberapa frame.

In [ ]:
# ambil beberapa frame sample terus plot
cap = cv2.VideoCapture(VIDEO_PATH)
total_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# pilih 6 frame yang tersebar
sample_idx = np.linspace(0, min(total_frame-1, 300), 6, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Hasil Deteksi YOLO di Beberapa Frame', fontsize=14)

for ax, idx in zip(axes.flatten(), sample_idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ret, frame = cap.read()
    if not ret:
        continue
    annotated, dets = detect_vehicles(frame, model)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {idx} | {len(dets)} kendaraan")
    ax.axis('off')

cap.release()
plt.tight_layout()
plt.show()

### Hasil Tracking
Rangkuman sederhana.

In [ ]:
if counts:
    print(f"Total frame: {len(counts)}")
    print(f"Total kendaraan terdeteksi: {np.sum(counts)}")
    print(f"Rata2 per frame: {np.mean(counts):.1f}")
    print(f"Paling banyak: {np.max(counts)} di 1 frame")
    print(f"Paling sedikit: {np.min(counts)}")

    # plot distribusi
    plt.figure(figsize=(10, 4))
    plt.plot(counts, alpha=0.7)
    plt.axhline(np.mean(counts), color='r', linestyle='--', label=f'rata2: {np.mean(counts):.1f}')
    plt.xlabel('Frame')
    plt.ylabel('Jumlah Kendaraan')
    plt.title('Distribusi Kendaraan per Frame')
    plt.legend()
    plt.show()

---
# Bagian 2: Teori
---

### Soal 2a: Apa itu AI? Contohnya?

AI (Artificial Intelligence) = sistem yang bisa ngerjain tugas yang normally butuh kecerdasan manusia. Intinya gimana caranya mesin bisa belajar dari data, ngambil keputusan, atau ngelakuin sesuatu yang keliatannya "pintar".

**Contoh sehari-hari:**

1. **Rekomendasi YouTube/Netflix** — AI nonton history kita, terus nebak video apa yang mungkin kita suka. Pake collaborative filtering.

2. **Google Assistant/Siri/ChatGPT** — NLP (Natural Language Processing) buat ngerti apa yang kita omongin, trus nyari jawaban atau ngejalanin perintah.

### Soal 2b: Supervised vs Unsupervised Learning

**Supervised Learning:**
- Pake data berlabel (ada jawabannya)
- Tujuan: prediksi output dari input
- Contoh: klasifikasi email spam (data udah dilabeli spam/ga spam, model belajar bedainnya)
- Algoritma: Linear Regression, Random Forest, SVM

**Unsupervised Learning:**
- Pake data TANPA label
- Tujuan: nyari pola/kelompok dalam data
- Contoh: segmentasi customer (model ngelompokin sendiri pelanggan berdasarkan perilaku belanja)
- Algoritma: K-Means, DBSCAN, PCA

Simple-nya: supervised = ada guru yang kasih jawaban, unsupervised = murid nyari pola sendiri.

### Soal 3a: Feature dalam ML

Feature = variabel atau ciri-ciri dari data yang dipake model buat belajar. Intinya input yang dikasih ke model.

Misal prediksi harga rumah, feature-nya:
- luas tanah, jumlah kamar, lokasi, tahun dibangun, jarak ke pusat kota

**Kenapa pemilihan feature penting?**

1. **Relevansi** — feature yang ga relevan cuma jadi noise, bikin model bingung
2. **Curse of dimensionality** — makin banyak feature, makin banyak data yang dibutuhin. Feature sampah bikin dimensi gede tapi ga nambah info
3. **Interpretability** — makin dikit feature yang bermakna, makin gampang dijelasin
4. **Komputasi** — feature dikit = training lebih cepet
5. **Overfitting** — feature kebanyakan bisa bikin model hafal noise bukan pola

Teknik seleksi: feature importance (dari tree), PCA, mutual information, RFE.

### Soal 3b: Fine-tuning itu apa?

Fine-tuning = transfer learning. Model yang udah dilatih di dataset gede dipake lagi (dilatih ulang dikit) buat tugas spesifik yang mirip.

Caranya:
- Ambil model pre-trained (udah jago ngeliat pola umum)
- Ganti layer terakhirnya sesuai tugas baru
- Train ulang dengan learning rate kecil (dikit aja, soalnya modelnya udah pinter)

**Contoh: deteksi sampah pesisir pake YOLO**

YOLO yang udah di-pre-train di COCO ga tau kelas "sampah plastik" atau "botol di pantai". Tapi karena YOLO udah jago deteksi objek secara umum, kita tinggal fine-tune pake ratusan gambar sampah pesisir yang udah dilabelin. Dalam beberapa epoch, model udah bisa deteksi sampah. Jauh lebih efisien daripada training dari 0 (yang butuh dataset gede banget).

Ini yang gw pake di TA dulu (DINO + CLIP buat deteksi sampah pesisir)— fine-tuning bikin hidup jauh lebih gampang.